<a href="https://colab.research.google.com/github/ja2faded/ds2002-fa26/blob/main/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [14]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'

RETRY_STATUSES = {429, 500, 502, 503, 504}



def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""

    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

#keep track of most recent error
    last = None


    # loop up to `tries` times
    for attempt in range(tries):
        try:
           #  request with a timeout
          r = requests.get(BASE, params=params, timeout=15)

          # return pd.DataFrame(r.json()['daily']) on 200
          if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])

          #raise on a status that is not in RETRY_STATUSES
          if r.status_code not in RETRY_STATUSES:
            raise RuntimeError(f'not retryable: {r.status_code} {r.text[:200]}')

            #if status is retryable
          last = f'status {r.status_code}'

        except requests.RequestException as e:
          last = repr(e)

        #sleep when another attempt remains (increasing each time) and try again; basedelay set to .5
        if attempt < tries - 1:
          delay = 0.5 * 2 ** attempt
          time.sleep(delay)

    # raise if every attempt failed
    raise RuntimeError(f'gave up after {tries} attempts: {last}')


### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [15]:
used_fallback = False

try:

  # call get_weather once for a range that spans both dates
    wx = get_weather('2026-09-05', '2026-10-03')

    # then keep just the two dates you care about
    #copy ensures will not affect original
    wx = wx[wx['time'].isin(['2026-09-05' , '2026-10-03'])].copy()

except Exception as e:

  used_fallback = True

  print('Request failed: ', e)
  print('Using fallback')



### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [16]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
#copy ensures will not affect original
if used_fallback:
  wx = FALLBACK.copy()

  # add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
  wx['source'] = 'fallback sample'

else:
  wx['source'] = 'open-meteo'


wx


,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [17]:
#rename columns
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

#convert date from text to real data time column
wx['date'] = pd.to_datetime(wx['date'])

wx


,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [18]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [19]:
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

difference = wetter['precip_mm'] - drier['precip_mm']

print(
    f"{wetter['date']} was wetter than " f"{drier['date']} by {difference} mm"
)

2026-10-03 00:00:00 was wetter than 2026-09-05 00:00:00 by 12.2 mm


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [20]:
from datetime import datetime

pulled = datetime.now().strftime('%Y-%m-%d %H:%M')
start_date = '2026-09-05'
end_date = '2026-10-03'
lat = 38.03
lon = -78.51

endpoint = BASE

print('endpoint:', BASE)
print('coordinates: ',lat, lon )
print('Pulled: ', pulled)
print('Date range: ', start_date, ' to ', end_date)

if used_fallback:
  source = 'fallback sample'
else:
  source = 'open-meteo'

print('Source: ', source)




endpoint: https://archive-api.open-meteo.com/v1/archive
coordinates:  38.03 -78.51
Pulled:  2026-10-10 18:48
Date range:  2026-09-05  to  2026-10-03
Source:  open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame rather than raising an error, it would appear that the merge worked when, in reality, the get_weather function returned an empty DataFrame and no weather data was collected.

Using left merge onto the 200 days of sales, all 200 rows would be shown, but there would be NA (missing) values because there would be no matching values in the empty weather DataFrame. A chart looking at rain against revenue would be empty (at least for the rain column) since missing values are dropped, and if the missing values were filled with 0, it would appear that rain had no effect on revenue.

It is possible that no one caught this error because when the get_weather function and the merge was ran, no error was outputted, leading the developer to assume that weather was collected and the merge happened smoothly (since after the merge the table still has 200 rows)

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [21]:
#pull orlando weather for same range
orlando = get_weather(start_date, end_date, lat=28.54, lon=-81.38)

#keep only the two dates
orlando = orlando[orlando['time'].isin([start_date, end_date])].copy()

#rename columns to match
orlando = orlando.rename(columns = {
    'time':'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})


#convert date to datetime
orlando['date'] = pd.to_datetime(orlando['date'])

#label orlando rows
orlando['source'] = 'open-meteo'
orlando['city'] = "Orlando"

#label charlottesville row
charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'

#combine
both = pd.concat([orlando, charlottesville])
both




,date,temp_max,precip_mm,source,city
0,2026-09-05,32.1,3.9,open-meteo,Orlando
28,2026-10-03,30.4,14.5,open-meteo,Orlando
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
28,2026-10-03,22.0,14.7,open-meteo,Charlottesville
